In [0]:
-- High Level Sales Analysis

-- 1: What was the total quantity sold for all products

-- total sold
SELECT
sum(qty) as total_sold
FROM sales;

-- total sold by product
SELECT pd.product_name,
SUM(s.qty) as total_sold
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.product_name
ORDER BY SUM(s.qty) desc;

-- 2: What was the total generated revenue for all products before discounts

-- total revenue
SELECT
sum(qty * price) as total_rev
FROM sales;

-- total revenue by product
SELECT pd.product_name,
SUM(s.qty * s.price) as rev_gen
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.product_name
ORDER BY SUM(s.qty * s.price) DESC;

-- 3: What was the total discount amount for all products

-- total discount
SELECT
SUM(qty * price * discount) / 100 as total_discount
FROM sales;

-- total discount by product
SELECT
pd.product_name,
SUM(s.qty * s.price * s.discount) / 100 as total_discount
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.product_name
ORDER BY (SUM(s.qty * s.price * s.discount) / 100) desc;




-- Transaction Analysis

-- 1: How many unique transactions were there?
SELECT COUNT(DISTINCT txn_id) as unique_transaction_ids FROM sales;

-- 2: What is the average unique products purchased in each transaction?
WITH cte as (SELECT count(distinct prod_id) as uniq_prods_purchased FROM sales
GROUP BY txn_id)
SELECT ROUND(AVG(uniq_prods_purchased), 0) as average_uniq_prods FROM cte;

-- 3: What are the 25th, 50th and 75th percentile values for the revenue per transaction?
WITH cte as (SELECT distinct txn_id,
SUM(qty * price * ((100 - discount) / 100)) as total_revenue
FROM sales
GROUP BY txn_id)

SELECT distinct percentile_cont(0.25) WITHIN GROUP (ORDER BY total_revenue) OVER () as p25,
percentile_cont(0.5) WITHIN GROUP (ORDER BY total_revenue) OVER () as p50,
percentile_cont(0.75) WITHIN GROUP (ORDER BY total_revenue) OVER () as p75
FROM cte

-- 4: What is the average discount value per transaction?
WITH cte as (SELECT 
sum(discount) as discount, 
txn_id FROM sales
GROUP BY txn_id)

SELECT ROUND(AVG(discount), 2) as average_discount FROM cte;

SELECT SUM(qty * price * (cast(discount as decimal(10,2)) /100) ) / COUNT (distinct txn_id) AS average_discount 
FROM sales ;

-- 5: What is the percentage split of all transactions for members vs non-members?
WITH transactions as (SELECT 
SUM(CASE WHEN member = true THEN 1 END) as total_members,
SUM(CASE WHEN member = false THEN 1 END) as total_non_members,
COUNT(member) as total_transactions
FROM sales)

SELECT 
ROUND(total_members / total_transactions * 100, 2) as member_percentage,
ROUND(total_non_members / total_transactions * 100, 2) as non_member_percentage
FROM transactions;

-- 6: What is the average revenue for member transactions and non-member transactions?
WITH cte as (SELECT
distinct txn_id,
member,
SUM(price * qty * ((100 - discount) / 100)) as revenue
FROM sales
GROUP BY txn_id, member)

SELECT distinct member,
ROUND(AVG(revenue) OVER (PARTITION BY member), 2) as avg_revenue
FROM cte;



